# SANAD — Stage 3
## LLM Generation, Grounding Evaluation & Citation Verification

This notebook rebuilds Stage 3 using the **final Stage 2 pipeline**:

**Saudi Legal MOJ → BGE-M3 Dense Retrieval → Dense Top-10 → Qwen3-Reranker-0.6B → Top-3 legal articles → Stage 3**

The final reranker experiment selected **Qwen3-Reranker-0.6B** for the Top-3 context because it achieved the strongest **Recall@3** and **MRR@10** in the final controlled comparison.

### Stage 3 generator comparison
- **humain-ai/ALLaM-7B-Instruct-preview**
- **Qwen/Qwen3-8B**

Both models receive the **same user query and the same Qwen-reranked Top-3 legal context**.

Stage 3 does not retrieve new laws. Its role is to transform the retrieved legal evidence into a grounded Arabic answer with accurate legal citations.

## 1. Experiment Goal

The goal is to compare ALLaM and Qwen3 as the final generation model for SANAD.

The comparison measures:

1. **Correctness** — whether the answer correctly addresses the legal question using the gold legal article.
2. **Faithfulness** — whether the answer's legal claims are supported by the retrieved Top-3 context.
3. **Citation Accuracy** — whether the answer cites the correct law and article and whether cited references exist in the retrieved context.
4. **Hallucination** — whether the answer introduces unsupported legal claims or fabricated citations.
5. **Arabic Quality** — clarity, coherence, professionalism, and Arabic language quality.
6. **Generation Time** — average inference time per query under the same runtime conditions.

The same **60-query gold test set** used in Stage 2 is retained to keep the complete SANAD evaluation traceable from retrieval to generation.

## 2. Why Stage 3 Was Rebuilt

The earlier Stage 3 design assumed that Hybrid Retrieval would produce the candidate set.

The final Stage 2 experiments changed that decision:

- **BGE-M3 Dense Retrieval** was retained instead of the tested Hybrid RRF configuration.
- Dense Top-10 candidates were reranked with **BGE-reranker-v2-m3** and **Qwen3-Reranker-0.6B**.
- **Qwen3-Reranker-0.6B** was selected for the final Top-3 context.

Therefore, this Stage 3 notebook must use the **new `augmented_context_FINAL.csv` generated with Qwen reranking**, not any older Hybrid/BGE context file.

## 3. Controlled Comparison Design

To keep the ALLaM vs Qwen3 comparison fair:

- Same 60 queries.
- Same Qwen-reranked Top-3 legal articles.
- Same generation prompt.
- No extra retrieval inside Stage 3.
- Same Colab GPU runtime.
- Models loaded sequentially to fit GPU memory.
- Same 4-bit inference setup.
- Same maximum output length.
- Deterministic decoding (`do_sample=False`) to reduce run-to-run variation.
- Qwen3 thinking mode is disabled so both models are compared on direct answer generation.

### Important evaluation rule

Because the reranker Recall@3 is below 100%, the gold article is not available in the final Top-3 for every query.

Therefore Stage 3 reports:

- **End-to-end citation accuracy** across all 60 queries.
- **Conditional citation accuracy** only when the gold article exists in the Top-3 context.

This separates upstream retrieval/reranking errors from generation errors.

## 4. Required Input Files

Upload:

1. `augmented_context_FINAL.csv`  
   Final context file generated after selecting **Qwen** in Experiment C.

2. `saudi_legal_moj_clean.parquet`  
   Used only to recover the gold legal article text for evaluation.

Expected columns in `augmented_context_FINAL.csv`:

- `query_id`
- `query_type`
- `query`
- `gold_law`
- `gold_article`
- `selected_reranker`
- `augmented_context`

## 5. Install libraries

In [1]:
!pip install -q -U "transformers>=4.51.0" accelerate bitsandbytes sentencepiece pyarrow pandas tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 17.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 7.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but you have pandas 3.0.6 which is incompatible.
das

In [2]:
!pip install pandas==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 82.9 MB/s eta 0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 3.0.6
    Uninstalling pandas-3.0.6:
      Successfully uninstalled pandas-3.0.6


## 6. Imports, seed, GPU check, and checkpoint folder

In [4]:
import os
import re
import gc
import json
import time
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    set_seed,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
set_seed(SEED)

assert torch.cuda.is_available(), (
    "GPU is required. In Colab: Runtime → Change runtime type → T4 GPU."
)

print("GPU:", torch.cuda.get_device_name(0))

USE_DRIVE_CHECKPOINT = True

if USE_DRIVE_CHECKPOINT:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK_DIR = Path("/content/drive/MyDrive/SANAD_Stage3")
else:
    WORK_DIR = Path("/content/SANAD_Stage3")

WORK_DIR.mkdir(parents=True, exist_ok=True)
print("Working directory:", WORK_DIR)

GPU: Tesla T4
Mounted at /content/drive
Working directory: /content/drive/MyDrive/SANAD_Stage3


## 7. Upload or restore the required files

In [5]:
from google.colab import files

required_files = [
    "augmented_context_FINAL.csv",
    "saudi_legal_moj_clean.parquet",
]

for filename in required_files:
    content_path = Path("/content") / filename
    saved_path = WORK_DIR / filename

    if (not content_path.exists()) and saved_path.exists():
        shutil.copy(saved_path, content_path)
        print("Restored:", filename)

missing = [f for f in required_files if not (Path("/content") / f).exists()]

if missing:
    print("Please upload:", missing)
    files.upload()

for filename in required_files:
    src = Path("/content") / filename
    assert src.exists(), f"Missing required file: {filename}"

    dst = WORK_DIR / filename
    if not dst.exists():
        shutil.copy(src, dst)

print("✅ Required files are ready.")

✅ Required files are ready.


## 8. Load and validate the final Stage 2 context

This cell stops the notebook if the uploaded context is not the final **Qwen-reranked** context.

In [6]:
CONTEXT_PATH = "/content/augmented_context_FINAL.csv"
KB_PATH = "/content/saudi_legal_moj_clean.parquet"

context_df = pd.read_csv(CONTEXT_PATH)
kb_df = pd.read_parquet(KB_PATH).reset_index(drop=True)

required_columns = {
    "query_id",
    "query_type",
    "query",
    "gold_law",
    "gold_article",
    "selected_reranker",
    "augmented_context",
}

missing_cols = required_columns - set(context_df.columns)
assert not missing_cols, f"Missing columns: {missing_cols}"
assert len(context_df) == 60, f"Expected 60 queries, found {len(context_df)}."

rerankers = set(
    context_df["selected_reranker"]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Rows:", len(context_df))
print("Selected reranker values:", rerankers)
print("Knowledge-base rows:", len(kb_df))

assert rerankers == {"Qwen"}, (
    "This is not the final Qwen-reranked context file. "
    "Rebuild augmented_context_FINAL.csv from Experiment C with WINNER='Qwen'."
)

print("✅ Final Qwen-reranked Stage 2 context verified.")

Rows: 60
Selected reranker values: {'Qwen'}
Knowledge-base rows: 4226
✅ Final Qwen-reranked Stage 2 context verified.


## 9. Parse Top-3 citations and measure context sufficiency

Before generation, we identify whether the gold law/article is present in the final Top-3.

A generator cannot correctly cite a gold article that was never supplied by retrieval/reranking.

In [7]:
ARABIC_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED\u0640]")

def normalize_label(value):
    s = str(value or "").strip()
    s = ARABIC_DIACRITICS.sub("", s)
    s = s.replace("أ", "ا").replace("إ", "ا").replace("آ", "ا").replace("ٱ", "ا")
    s = s.replace("ى", "ي").replace("ؤ", "و").replace("ئ", "ي")
    s = re.sub(r"[\s\-_–—:،,؛;.!؟?()\[\]{}]+", "", s)
    return s

def pair_key(law, article):
    return normalize_label(law) + "||" + normalize_label(article)

def extract_retrieved_pairs(augmented_context):
    text = str(augmented_context or "")
    matches = re.findall(
        r"Law:\s*(.*?)\s*\nArticle:\s*(.*?)\s*\nText:",
        text,
        flags=re.DOTALL,
    )
    return [(law.strip(), article.strip()) for law, article in matches]

context_df["retrieved_pairs"] = context_df["augmented_context"].apply(extract_retrieved_pairs)
context_df["retrieved_pair_keys"] = context_df["retrieved_pairs"].apply(
    lambda xs: [pair_key(a, b) for a, b in xs]
)
context_df["gold_key"] = context_df.apply(
    lambda r: pair_key(r["gold_law"], r["gold_article"]),
    axis=1,
)
context_df["context_has_gold"] = context_df.apply(
    lambda r: r["gold_key"] in r["retrieved_pair_keys"],
    axis=1,
)

print("Top-3 parsed successfully:",
      sum(len(x) == 3 for x in context_df["retrieved_pairs"]),
      "/ 60")

print("Gold article available in Top-3:",
      int(context_df["context_has_gold"].sum()),
      "/ 60")

print("Top-3 gold coverage:",
      round(context_df["context_has_gold"].mean(), 4))

display(
    context_df[
        ["query_id", "query_type", "gold_law", "gold_article", "context_has_gold"]
    ].head()
)

Top-3 parsed successfully: 60 / 60
Gold article available in Top-3: 54 / 60
Top-3 gold coverage: 0.9


,query_id,query_type,gold_law,gold_article,context_has_gold
0,Q001,formal_direct,نظام التكاليف القضائية,المادة الثانية,True
1,Q002,formal_paraphrase,نظام التكاليف القضائية,المادة الثانية,True
2,Q003,informal,نظام التكاليف القضائية,المادة الثانية,False
3,Q004,formal_direct,نظام التكاليف القضائية,المادة الثالثة,True
4,Q005,formal_paraphrase,نظام التكاليف القضائية,المادة الثالثة,True


## 10. Recover the gold legal article text

The gold article text is used **only for evaluation**, never as generation input.

This preserves the realism of the RAG pipeline while giving the evaluator a trusted reference.

In [8]:
kb_df["_law_norm"] = kb_df["law_name"].map(normalize_label)
kb_df["_article_norm"] = kb_df["article_number"].map(normalize_label)
kb_df["_gold_key"] = kb_df["_law_norm"] + "||" + kb_df["_article_norm"]

gold_text_map = (
    kb_df.drop_duplicates("_gold_key")
    .set_index("_gold_key")["text"]
    .astype(str)
    .to_dict()
)

context_df["gold_text"] = context_df["gold_key"].map(gold_text_map)

missing_gold_text = context_df["gold_text"].isna().sum()
print("Missing gold article texts:", missing_gold_text)

assert missing_gold_text == 0, (
    "Some gold articles could not be matched in the cleaned knowledge base."
)

print("✅ Gold reference texts recovered.")

Missing gold article texts: 0
✅ Gold reference texts recovered.


## 11. Shared SANAD generation prompt

Each model must:

- answer in Arabic;
- use only the retrieved legal context;
- avoid outside legal knowledge;
- abstain if the context is insufficient;
- cite law and article exactly as they appear in the context;
- return valid JSON for reproducible evaluation.

### Required output

```json
{
  "answer_ar": "Arabic grounded answer",
  "citations": [
    {"law": "اسم النظام", "article": "رقم المادة"}
  ],
  "insufficient_context": false
}
```

In [9]:
def build_generation_prompt(row):
    query = str(row["query"]).strip()
    context = str(row["augmented_context"]).strip()

    return f"""
أنت مساعد قانوني سعودي ضمن نظام SANAD.

مهمتك الإجابة عن سؤال المستخدم اعتماداً فقط على السياق القانوني المسترجع أدناه.

قواعد إلزامية:
1) لا تستخدم أي معلومة قانونية من خارج السياق المسترجع.
2) لا تخترع نظاماً أو مادة أو رقماً غير موجود في السياق.
3) اذكر اسم النظام ورقم المادة كما ظهرا في السياق.
4) إذا كان السياق لا يكفي للإجابة بثقة، صرّح بذلك واجعل insufficient_context = true.
5) يجب أن تكون الإجابة باللغة العربية، واضحة ومختصرة ومهنية.
6) لا تكتب أي نص خارج JSON.
7) أعد JSON صالحاً تماماً بهذا الشكل:
{{
  "answer_ar": "الإجابة العربية",
  "citations": [
    {{"law": "اسم النظام", "article": "رقم المادة"}}
  ],
  "insufficient_context": false
}}

سؤال المستخدم:
{query}

السياق القانوني المسترجع:
{context}
""".strip()

context_df["generation_prompt"] = context_df.apply(build_generation_prompt, axis=1)

print(context_df.loc[0, "generation_prompt"][:2200])

أنت مساعد قانوني سعودي ضمن نظام SANAD.

مهمتك الإجابة عن سؤال المستخدم اعتماداً فقط على السياق القانوني المسترجع أدناه.

قواعد إلزامية:
1) لا تستخدم أي معلومة قانونية من خارج السياق المسترجع.
2) لا تخترع نظاماً أو مادة أو رقماً غير موجود في السياق.
3) اذكر اسم النظام ورقم المادة كما ظهرا في السياق.
4) إذا كان السياق لا يكفي للإجابة بثقة، صرّح بذلك واجعل insufficient_context = true.
5) يجب أن تكون الإجابة باللغة العربية، واضحة ومختصرة ومهنية.
6) لا تكتب أي نص خارج JSON.
7) أعد JSON صالحاً تماماً بهذا الشكل:
{
  "answer_ar": "الإجابة العربية",
  "citations": [
    {"law": "اسم النظام", "article": "رقم المادة"}
  ],
  "insufficient_context": false
}

سؤال المستخدم:
ما الدعاوى والطلبات التي لا تسري عليها أحكام التكاليف القضائية؟

السياق القانوني المسترجع:
User Query:
ما الدعاوى والطلبات التي لا تسري عليها أحكام التكاليف القضائية؟

[Retrieved Article 1]
Law: نظام التكاليف القضائية
Article: المادة الثانية
Text: المادة الثانية
تسري أحكام النظام على جميع الدعاوى والطلبات التي تقدم إلى المحاكم»

## 12. Model and generation configuration

### Compared generators
- `humain-ai/ALLaM-7B-Instruct-preview`
- `Qwen/Qwen3-8B`

### Shared settings
- 4-bit NF4 loading.
- Same prompt content.
- Same 60 queries.
- Same maximum input budget.
- Same maximum output budget.
- Deterministic generation.
- Sequential execution.
- Qwen3 thinking disabled.

The shared input limit is conservative because ALLaM has the smaller context window.

In [10]:
ALLAM_MODEL_ID = "humain-ai/ALLaM-7B-Instruct-preview"
QWEN_MODEL_ID = "Qwen/Qwen3-8B"

MAX_INPUT_TOKENS = 3400
MAX_NEW_TOKENS = 350

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

def load_generator(model_id, revision=None):
    print("Loading:", model_id)

    tokenizer = AutoTokenizer.from_pretrained(
        model_id,
        revision=revision,
        use_fast=True,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        revision=revision,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    model.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    print("✅ Loaded:", model_id)
    return tokenizer, model

def release_model(tokenizer=None, model=None):
    if model is not None:
        del model
    if tokenizer is not None:
        del tokenizer

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()

def format_for_model(tokenizer, model_id, prompt):
    messages = [{"role": "user", "content": prompt}]

    kwargs = {
        "tokenize": False,
        "add_generation_prompt": True,
    }

    if model_id == QWEN_MODEL_ID:
        kwargs["enable_thinking"] = False

    try:
        return tokenizer.apply_chat_template(messages, **kwargs)
    except TypeError:
        kwargs.pop("enable_thinking", None)
        return tokenizer.apply_chat_template(messages, **kwargs)
    except Exception:
        return prompt

def generate_one(tokenizer, model, model_id, prompt, max_new_tokens=MAX_NEW_TOKENS):
    formatted = format_for_model(tokenizer, model_id, prompt)

    inputs = tokenizer(
        formatted,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_TOKENS,
    )

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    input_tokens = int(inputs["input_ids"].shape[-1])
    input_truncated = input_tokens >= MAX_INPUT_TOKENS

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    start = time.perf_counter()

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    elapsed = time.perf_counter() - start

    new_tokens = output_ids[0][inputs["input_ids"].shape[-1]:]
    response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return response, elapsed, input_tokens, input_truncated

## 13. Resumable generation

Every completed query is checkpointed. If Colab disconnects, rerun the notebook and the model continues from remaining query IDs.

In [11]:
def run_generator(
    model_label,
    model_id,
    context_df,
    checkpoint_path,
    revision=None,
):
    checkpoint_path = Path(checkpoint_path)

    if checkpoint_path.exists():
        previous = pd.read_csv(checkpoint_path)
        results = previous.to_dict("records")
        completed_ids = set(previous["query_id"].astype(str))
        print(f"Resuming {model_label}: {len(completed_ids)}/60 completed.")
    else:
        results = []
        completed_ids = set()

    tokenizer, model = load_generator(model_id, revision=revision)

    try:
        for _, row in tqdm(
            context_df.iterrows(),
            total=len(context_df),
            desc=f"{model_label} generation",
        ):
            qid = str(row["query_id"])

            if qid in completed_ids:
                continue

            response, elapsed, input_tokens, was_truncated = generate_one(
                tokenizer,
                model,
                model_id,
                row["generation_prompt"],
            )

            results.append({
                "query_id": qid,
                "model": model_label,
                "raw_response": response,
                "generation_time_s": elapsed,
                "input_tokens": input_tokens,
                "input_truncated": was_truncated,
            })

            pd.DataFrame(results).to_csv(
                checkpoint_path,
                index=False,
                encoding="utf-8-sig",
            )

    finally:
        release_model(tokenizer, model)

    result_df = pd.read_csv(checkpoint_path)
    print(f"✅ {model_label}: {len(result_df)}/60 completed.")
    return result_df

## 14. Optional smoke test

Use this only to verify that the runtime and model loading work. Keep it disabled during the final experiment.

In [12]:
RUN_SMOKE_TEST = False

if RUN_SMOKE_TEST:
    tok, mdl = load_generator(ALLAM_MODEL_ID, revision="v2")

    for i in range(2):
        row = context_df.iloc[i]
        response, elapsed, tokens, truncated = generate_one(
            tok,
            mdl,
            ALLAM_MODEL_ID,
            row["generation_prompt"],
        )
        print("=" * 80)
        print(row["query_id"], "|", elapsed, "seconds")
        print(response)

    release_model(tok, mdl)

# 15. Generate answers with ALLaM-7B-Instruct-preview

In [14]:
ALLAM_CHECKPOINT = WORK_DIR / "allam_generation_checkpoint.csv"

allam_df = run_generator(
    model_label="ALLaM-7B-Instruct-preview",
    model_id=ALLAM_MODEL_ID,
    context_df=context_df,
    checkpoint_path=ALLAM_CHECKPOINT,
    revision=None,
)

display(allam_df.head(2))

Loading: humain-ai/ALLaM-7B-Instruct-preview


config.json:   0%|          | 0.00/686 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B / 1.23MB            

tokenizer.model: downloading bytes:           |  0.00B            

...

✅ Loaded: humain-ai/ALLaM-7B-Instruct-preview


ALLaM-7B-Instruct-preview generation:   0%|          | 0/60 [00:00<?, ?it/s]

✅ ALLaM-7B-Instruct-preview: 60/60 completed.


,query_id,model,raw_response,generation_time_s,input_tokens,input_truncated
0,Q001,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الدعاوى الجزائية العامة، ال...",10.178966,759,False
1,Q002,ALLaM-7B-Instruct-preview,"{\n ""answer_ar"": ""الحالات المستثناة من تطبيق ...",13.186868,751,False


Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


# 16. Generate answers with Qwen3-8B

In [15]:
QWEN_CHECKPOINT = WORK_DIR / "qwen3_generation_checkpoint.csv"

qwen_df = run_generator(
    model_label="Qwen3-8B",
    model_id=QWEN_MODEL_ID,
    context_df=context_df,
    checkpoint_path=QWEN_CHECKPOINT,
)

display(qwen_df.head(2))

Loading: Qwen/Qwen3-8B


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

✅ Loaded: Qwen/Qwen3-8B


Qwen3-8B generation:   0%|          | 0/60 [00:00<?, ?it/s]

✅ Qwen3-8B: 60/60 completed.


,query_id,model,raw_response,generation_time_s,input_tokens,input_truncated
0,Q001,Qwen3-8B,"{\n ""answer_ar"": ""الدعاوى والطلبات التي لا تس...",19.172787,1009,False
1,Q002,Qwen3-8B,"{\n ""answer_ar"": ""الحالات المستثناة من تطبيق ...",30.649801,996,False


## 17. Parse model outputs

The parser accepts valid JSON directly and also handles a JSON object wrapped in Markdown fences.

In [16]:
def extract_json_object(text):
    s = str(text or "").strip()

    s = re.sub(r"^```(?:json)?\s*", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s*```$", "", s)

    try:
        return json.loads(s), True
    except Exception:
        pass

    start = s.find("{")
    end = s.rfind("}")

    if start != -1 and end > start:
        candidate = s[start:end + 1]
        try:
            return json.loads(candidate), True
        except Exception:
            pass

    return {
        "answer_ar": s,
        "citations": [],
        "insufficient_context": None,
    }, False

def normalize_citations(value):
    if not isinstance(value, list):
        return []

    out = []

    for item in value:
        if not isinstance(item, dict):
            continue

        law = str(item.get("law", "")).strip()
        article = str(item.get("article", "")).strip()

        if law and article:
            out.append({"law": law, "article": article})

    return out

def parse_generation_df(gen_df):
    rows = []

    for _, r in gen_df.iterrows():
        obj, valid = extract_json_object(r["raw_response"])

        rows.append({
            **r.to_dict(),
            "valid_json": valid,
            "answer_ar": str(obj.get("answer_ar", "")).strip(),
            "citations": normalize_citations(obj.get("citations", [])),
            "insufficient_context": obj.get("insufficient_context", None),
        })

    return pd.DataFrame(rows)

allam_parsed = parse_generation_df(allam_df)
qwen_parsed = parse_generation_df(qwen_df)

generations_df = pd.concat(
    [allam_parsed, qwen_parsed],
    ignore_index=True,
)

print("Generation rows:", len(generations_df))
display(
    generations_df[
        ["query_id", "model", "valid_json", "generation_time_s", "answer_ar"]
    ].head()
)

Generation rows: 120


,query_id,model,valid_json,generation_time_s,answer_ar
0,Q001,ALLaM-7B-Instruct-preview,True,10.178966,الدعاوى الجزائية العامة، الدعاوى التأديبية، ال...
1,Q002,ALLaM-7B-Instruct-preview,True,13.186868,الحالات المستثناة من تطبيق نظام التكاليف القضا...
2,Q003,ALLaM-7B-Instruct-preview,True,15.644944,الدعاوى الجزائية الخاصة، الدعاوى التي تُنظر لد...
3,Q004,ALLaM-7B-Instruct-preview,True,10.653370,الحد الأعلى للتكاليف القضائية المفروضة على الد...
4,Q005,ALLaM-7B-Instruct-preview,True,22.305828,تُقدّر التكاليف القضائية للدعوى المحددة القيمة...


# 18. Automatic Citation Verification

This section is deterministic and does not use another LLM.

Metrics:

- **Citation Coverage** — answer contains at least one citation.
- **Citation Precision vs Context** — proportion of cited law/article pairs that exist in retrieved Top-3.
- **Gold Citation Hit** — answer cites the gold law/article.
- **Citation Hallucination Rate** — proportion of citations not present in retrieved Top-3.

A citation that is not in the supplied context counts as ungrounded even if it might be legally plausible from model memory.

In [17]:
base_cols = [
    "query_id",
    "query_type",
    "query",
    "gold_law",
    "gold_article",
    "gold_key",
    "gold_text",
    "retrieved_pairs",
    "retrieved_pair_keys",
    "context_has_gold",
    "augmented_context",
]

eval_df = generations_df.merge(
    context_df[base_cols],
    on="query_id",
    how="left",
)

def citation_metrics(row):
    citations = row["citations"]

    cited_keys = [
        pair_key(c.get("law", ""), c.get("article", ""))
        for c in citations
    ]

    retrieved_keys = set(row["retrieved_pair_keys"])
    gold_key = row["gold_key"]

    valid_count = sum(key in retrieved_keys for key in cited_keys)
    total = len(cited_keys)

    return pd.Series({
        "citation_coverage": int(total > 0),
        "citation_precision_context": (valid_count / total) if total else 0.0,
        "citation_hallucination_rate": (
            (total - valid_count) / total if total else 0.0
        ),
        "gold_citation_hit": int(gold_key in cited_keys),
        "citation_count": total,
    })

citation_eval = eval_df.apply(citation_metrics, axis=1)
eval_df = pd.concat([eval_df, citation_eval], axis=1)

display(
    eval_df[
        [
            "query_id",
            "model",
            "context_has_gold",
            "citation_coverage",
            "citation_precision_context",
            "gold_citation_hit",
            "citation_hallucination_rate",
        ]
    ].head(10)
)

,query_id,model,context_has_gold,citation_coverage,citation_precision_context,gold_citation_hit,citation_hallucination_rate
0,Q001,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
1,Q002,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
2,Q003,ALLaM-7B-Instruct-preview,False,1.0,0.5,0.0,0.5
3,Q004,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
4,Q005,ALLaM-7B-Instruct-preview,True,1.0,0.0,0.0,1.0
5,Q006,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
6,Q007,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
7,Q008,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0
8,Q009,ALLaM-7B-Instruct-preview,False,1.0,1.0,0.0,0.0
9,Q010,ALLaM-7B-Instruct-preview,True,1.0,1.0,1.0,0.0


## 19. Automatic metric summary

Two gold-citation views are reported:

- **Gold Citation Accuracy — All Queries:** end-to-end pipeline performance.
- **Gold Citation Accuracy — Context Has Gold:** generator-only view on queries where retrieval supplied the gold article.

In [18]:
auto_summary_rows = []

for model_name, g in eval_df.groupby("model"):
    conditional = g[g["context_has_gold"] == True]

    auto_summary_rows.append({
        "Model": model_name,
        "Valid_JSON_Rate": g["valid_json"].mean(),
        "Citation_Coverage": g["citation_coverage"].mean(),
        "Citation_Precision_Context": g["citation_precision_context"].mean(),
        "Gold_Citation_Accuracy_All": g["gold_citation_hit"].mean(),
        "Gold_Citation_Accuracy_ContextHasGold": (
            conditional["gold_citation_hit"].mean()
            if len(conditional) else np.nan
        ),
        "Citation_Hallucination_Rate": g["citation_hallucination_rate"].mean(),
        "Avg_Generation_Time_s": g["generation_time_s"].mean(),
        "Input_Truncation_Rate": g["input_truncated"].mean(),
    })

automatic_summary_df = pd.DataFrame(auto_summary_rows)
display(automatic_summary_df)

,Model,Valid_JSON_Rate,Citation_Coverage,Citation_Precision_Context,Gold_Citation_Accuracy_All,Gold_Citation_Accuracy_ContextHasGold,Citation_Hallucination_Rate,Avg_Generation_Time_s,Input_Truncation_Rate
0,ALLaM-7B-Instruct-preview,0.95,0.95,0.819444,0.650000,0.722222,0.130556,9.572170,0.0
1,Qwen3-8B,1.00,1.00,0.850000,0.683333,0.759259,0.150000,15.592022,0.0


# 20. Content-Quality Evaluation with an Independent Judge

Citation checks alone cannot measure prose correctness or faithfulness.

A separate Arabic-capable evaluator model is used:

- `CohereLabs/aya-expanse-8b`

The judge receives the question, gold legal article, retrieved Top-3 context, and generated answer.

It must not use outside legal knowledge.

### Scores

**Correctness (1–5)**  
How correctly the answer addresses the question using the gold legal reference.

**Faithfulness (1–5)**  
How fully the answer's legal claims are supported by the retrieved context.

**Arabic Quality (1–5)**  
Clarity, coherence, and professional Arabic.

**Hallucination Flag (0/1)**  
1 if at least one material legal claim is unsupported by the retrieved context.

### Methodological note
LLM-as-a-judge is not treated as absolute ground truth. A blinded human-review file is also exported later.

## 21. Configure the independent judge

In [19]:
RUN_LLM_JUDGE = True
JUDGE_MODEL_ID = "CohereLabs/aya-expanse-8b"
JUDGE_MAX_NEW_TOKENS = 220

def build_judge_prompt(row):
    return f"""
أنت مقيم مستقل لإجابات نظام قانوني سعودي قائم على RAG.

قيّم الإجابة فقط باستخدام:
1) السؤال،
2) النص القانوني المرجعي الذهبي،
3) السياق المسترجع المعطى.

لا تستخدم معرفة قانونية خارجية.

أعد JSON فقط بهذا الشكل:
{{
  "correctness": 1,
  "faithfulness": 1,
  "arabic_quality": 1,
  "hallucination": 0,
  "reason": "سبب مختصر"
}}

تعريف الدرجات:
- correctness من 1 إلى 5: مدى صحة الإجابة بالنسبة للسؤال والنص القانوني الذهبي.
- faithfulness من 1 إلى 5: مدى دعم السياق المسترجع للادعاءات القانونية في الإجابة.
- arabic_quality من 1 إلى 5: وضوح وسلامة ومهنية اللغة العربية.
- hallucination = 1 إذا احتوت الإجابة ادعاءً قانونياً مادياً غير مدعوم بالسياق المسترجع، وإلا 0.

السؤال:
{row["query"]}

النص القانوني المرجعي الذهبي:
{row["gold_text"]}

السياق المسترجع:
{row["augmented_context"]}

الإجابة المراد تقييمها:
{row["answer_ar"]}
""".strip()

### Access Gated Model on Hugging Face

The `CohereLabs/aya-expanse-8b` model is hosted on Hugging Face as a *gated* model. This means you need to be logged in to Hugging Face and have accepted its terms of use to access it. If you haven't already, please visit the model page on Hugging Face and accept the terms.

To authenticate in Colab, you can use `huggingface_hub.login()`. It's recommended to store your Hugging Face token as a Colab secret named `HF_TOKEN` for secure access.

## 22. Run the judge with checkpointing

In [25]:
# ============================================================
# Cell 20 — Independent LLM Judge
# FIXED: No gated Hugging Face model / no HF login required
# ============================================================

# Override the gated Aya model with a public model
JUDGE_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
JUDGE_MAX_INPUT_TOKENS = 7600
JUDGE_MAX_NEW_TOKENS = 220


def parse_judge_json(text):
    obj, valid = extract_json_object(text)

    def safe_score(name):
        try:
            value = float(obj.get(name))
            return min(5.0, max(1.0, value))
        except Exception:
            return np.nan

    try:
        hallucination = int(obj.get("hallucination"))
        hallucination = 1 if hallucination else 0
    except Exception:
        hallucination = np.nan

    return {
        "judge_valid_json": valid,
        "correctness": safe_score("correctness"),
        "faithfulness": safe_score("faithfulness"),
        "arabic_quality": safe_score("arabic_quality"),
        "hallucination": hallucination,
        "judge_reason": str(obj.get("reason", "")).strip(),
    }


def run_judge(eval_df, checkpoint_path):
    checkpoint_path = Path(checkpoint_path)

    results = []
    completed = set()

    # --------------------------------------------------------
    # Resume previous checkpoint if available
    # --------------------------------------------------------
    if checkpoint_path.exists():
        try:
            previous = pd.read_csv(checkpoint_path)

            if {"query_id", "model"}.issubset(previous.columns):

                previous["query_id"] = previous["query_id"].astype(str)
                previous["model"] = previous["model"].astype(str)

                previous = previous.drop_duplicates(
                    subset=["query_id", "model"],
                    keep="last",
                )

                results = previous.to_dict("records")

                completed = set(
                    zip(
                        previous["query_id"],
                        previous["model"],
                    )
                )

                print(
                    f"Resuming judge: "
                    f"{len(completed)}/{len(eval_df)} completed."
                )

        except Exception as e:
            print(
                "⚠️ Previous judge checkpoint could not be loaded."
            )
            print("Starting judge from the beginning.")
            print("Reason:", e)

            results = []
            completed = set()

    # --------------------------------------------------------
    # Load public judge model
    # --------------------------------------------------------
    tokenizer = None
    model = None

    print(f"Loading Judge Model: {JUDGE_MODEL_ID}")

    try:
        tokenizer, model = load_generator(JUDGE_MODEL_ID)

        print("✅ Judge model loaded successfully.")

        # ----------------------------------------------------
        # Evaluate each generated answer
        # ----------------------------------------------------
        for _, row in tqdm(
            eval_df.iterrows(),
            total=len(eval_df),
            desc="Independent judging",
        ):

            key = (
                str(row["query_id"]),
                str(row["model"]),
            )

            # Skip rows already evaluated
            if key in completed:
                continue

            prompt = build_judge_prompt(row)

            # FIXED:
            # generate_one() does not accept max_input_tokens
            response, elapsed, input_tokens, was_truncated = generate_one(
                tokenizer,
                model,
                JUDGE_MODEL_ID,
                prompt,
                max_new_tokens=JUDGE_MAX_NEW_TOKENS,
            )

            parsed = parse_judge_json(response)

            results.append({
                "query_id": str(row["query_id"]),
                "model": str(row["model"]),

                "judge_raw_response": response,
                "judge_time_s": elapsed,
                "judge_input_tokens": input_tokens,
                "judge_input_truncated": was_truncated,

                **parsed,
            })

            # Save after every row
            pd.DataFrame(results).to_csv(
                checkpoint_path,
                index=False,
                encoding="utf-8-sig",
            )

            completed.add(key)

    finally:
        # ----------------------------------------------------
        # Safely release model from GPU memory
        # ----------------------------------------------------
        model = None
        tokenizer = None

        try:
            clear_cuda_cache()
        except Exception:
            import gc
            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    # --------------------------------------------------------
    # Final judge dataframe
    # --------------------------------------------------------
    judge_df = pd.DataFrame(results)

    if len(judge_df) == 0:
        raise RuntimeError(
            "The judge produced no results."
        )

    judge_df["query_id"] = judge_df["query_id"].astype(str)
    judge_df["model"] = judge_df["model"].astype(str)

    judge_df = judge_df.drop_duplicates(
        subset=["query_id", "model"],
        keep="last",
    )

    judge_df.to_csv(
        checkpoint_path,
        index=False,
        encoding="utf-8-sig",
    )

    return judge_df


# ============================================================
# Run Judge
# ============================================================

JUDGE_CHECKPOINT = WORK_DIR / "stage3_judge_checkpoint.csv"

if RUN_LLM_JUDGE:

    judge_df = run_judge(
        eval_df,
        JUDGE_CHECKPOINT,
    )

    # Remove old judge columns if Cell 20 is re-run
    judge_columns = [
        "judge_raw_response",
        "judge_time_s",
        "judge_input_tokens",
        "judge_input_truncated",
        "judge_valid_json",
        "correctness",
        "faithfulness",
        "arabic_quality",
        "hallucination",
        "judge_reason",
    ]

    eval_df = eval_df.drop(
        columns=[
            col
            for col in judge_columns
            if col in eval_df.columns
        ],
        errors="ignore",
    )

    eval_df["query_id"] = eval_df["query_id"].astype(str)
    eval_df["model"] = eval_df["model"].astype(str)

    eval_df = eval_df.merge(
        judge_df,
        on=["query_id", "model"],
        how="left",
        validate="one_to_one",
    )

    print("✅ Judge rows:", len(judge_df))

    if "judge_valid_json" in eval_df.columns:
        print(
            "✅ Valid Judge JSON:",
            f"{eval_df['judge_valid_json'].mean():.2%}"
        )

else:
    print("LLM judge skipped.")

Loading Judge Model: Qwen/Qwen2.5-7B-Instruct
Loading: Qwen/Qwen2.5-7B-Instruct


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

✅ Loaded: Qwen/Qwen2.5-7B-Instruct
✅ Judge model loaded successfully.


Independent judging:   0%|          | 0/120 [00:00<?, ?it/s]

✅ Judge rows: 120
✅ Valid Judge JSON: 100.00%


# 23. Final model comparison

Primary quality dimensions:

- Correctness
- Faithfulness
- Citation Accuracy
- Hallucination
- Arabic Quality

Generation time is a secondary efficiency metric.

In [26]:
final_rows = []

for model_name, g in eval_df.groupby("model"):
    conditional = g[g["context_has_gold"] == True]

    final_rows.append({
        "Model": model_name,
        "Correctness_1_5": g["correctness"].mean() if "correctness" in g else np.nan,
        "Faithfulness_1_5": g["faithfulness"].mean() if "faithfulness" in g else np.nan,
        "Citation_Accuracy_All": g["gold_citation_hit"].mean(),
        "Citation_Accuracy_ContextHasGold": (
            conditional["gold_citation_hit"].mean()
            if len(conditional) else np.nan
        ),
        "Citation_Precision_Context": g["citation_precision_context"].mean(),
        "Hallucination_Rate": g["hallucination"].mean() if "hallucination" in g else np.nan,
        "Citation_Hallucination_Rate": g["citation_hallucination_rate"].mean(),
        "Arabic_Quality_1_5": g["arabic_quality"].mean() if "arabic_quality" in g else np.nan,
        "Avg_Generation_Time_s": g["generation_time_s"].mean(),
        "Valid_JSON_Rate": g["valid_json"].mean(),
    })

final_comparison_df = pd.DataFrame(final_rows)
display(final_comparison_df)

,Model,Correctness_1_5,Faithfulness_1_5,Citation_Accuracy_All,Citation_Accuracy_ContextHasGold,Citation_Precision_Context,Hallucination_Rate,Citation_Hallucination_Rate,Arabic_Quality_1_5,Avg_Generation_Time_s,Valid_JSON_Rate
0,ALLaM-7B-Instruct-preview,4.100000,3.716667,0.650000,0.722222,0.819444,0.050000,0.130556,4.266667,9.572170,0.95
1,Qwen3-8B,3.933333,3.466667,0.683333,0.759259,0.850000,0.083333,0.150000,4.133333,15.592022,1.00


## 24. Results by query type

In [27]:
by_type_rows = []

for (query_type, model_name), g in eval_df.groupby(["query_type", "model"]):
    conditional = g[g["context_has_gold"] == True]

    by_type_rows.append({
        "Query_Type": query_type,
        "Model": model_name,
        "N": len(g),
        "Correctness_1_5": g["correctness"].mean() if "correctness" in g else np.nan,
        "Faithfulness_1_5": g["faithfulness"].mean() if "faithfulness" in g else np.nan,
        "Citation_Accuracy_All": g["gold_citation_hit"].mean(),
        "Citation_Accuracy_ContextHasGold": (
            conditional["gold_citation_hit"].mean()
            if len(conditional) else np.nan
        ),
        "Hallucination_Rate": g["hallucination"].mean() if "hallucination" in g else np.nan,
        "Arabic_Quality_1_5": g["arabic_quality"].mean() if "arabic_quality" in g else np.nan,
        "Avg_Generation_Time_s": g["generation_time_s"].mean(),
    })

by_query_type_df = pd.DataFrame(by_type_rows)
display(by_query_type_df)

,Query_Type,Model,N,Correctness_1_5,Faithfulness_1_5,Citation_Accuracy_All,Citation_Accuracy_ContextHasGold,Hallucination_Rate,Arabic_Quality_1_5,Avg_Generation_Time_s
0,formal_direct,ALLaM-7B-Instruct-preview,20,4.25,4.20,0.70,0.700000,0.00,4.4,9.025202
1,formal_direct,Qwen3-8B,20,4.40,4.00,0.80,0.800000,0.05,4.5,16.123505
2,formal_paraphrase,ALLaM-7B-Instruct-preview,20,4.15,3.55,0.70,0.777778,0.10,4.4,10.028999
3,formal_paraphrase,Qwen3-8B,20,3.55,2.95,0.60,0.666667,0.15,3.9,16.906399
4,informal,ALLaM-7B-Instruct-preview,20,3.90,3.40,0.55,0.687500,0.05,4.0,9.662309
5,informal,Qwen3-8B,20,3.85,3.45,0.65,0.812500,0.05,4.0,13.746163


## 25. Paired Bootstrap 95% Confidence Intervals

Both generators answer the same 60 queries, so model differences are paired.

If a 95% confidence interval includes zero, describe the difference cautiously instead of claiming a decisive advantage.

In [28]:
rng = np.random.default_rng(42)
N_BOOT = 5000

def paired_bootstrap_difference(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]

    if len(a) == 0:
        return np.nan, np.nan, np.nan

    observed = b.mean() - a.mean()
    diffs = np.empty(N_BOOT, dtype=float)
    n = len(a)

    for i in range(N_BOOT):
        idx = rng.integers(0, n, size=n)
        diffs[i] = b[idx].mean() - a[idx].mean()

    lo, hi = np.percentile(diffs, [2.5, 97.5])
    return observed, lo, hi

wide = eval_df.pivot(index="query_id", columns="model")

MODEL_A = "ALLaM-7B-Instruct-preview"
MODEL_B = "Qwen3-8B"

metrics_for_ci = [
    ("Correctness", "correctness"),
    ("Faithfulness", "faithfulness"),
    ("Gold Citation Hit", "gold_citation_hit"),
    ("Citation Precision", "citation_precision_context"),
    ("Arabic Quality", "arabic_quality"),
]

ci_rows = []

for label, col in metrics_for_ci:
    if col not in eval_df.columns:
        continue

    a = wide[col][MODEL_A]
    b = wide[col][MODEL_B]

    diff, lo, hi = paired_bootstrap_difference(a, b)

    ci_rows.append({
        "Metric": label,
        "Difference_Qwen_minus_ALLaM": diff,
        "95%_CI_Lower": lo,
        "95%_CI_Upper": hi,
    })

bootstrap_ci_df = pd.DataFrame(ci_rows)
display(bootstrap_ci_df)

,Metric,Difference_Qwen_minus_ALLaM,95%_CI_Lower,95%_CI_Upper
0,Correctness,-0.166667,-0.466667,0.133333
1,Faithfulness,-0.250000,-0.566667,0.083333
2,Gold Citation Hit,0.033333,-0.100000,0.166667
3,Citation Precision,0.030556,-0.094444,0.152778
4,Arabic Quality,-0.133333,-0.450000,0.183333


# 26. Blinded Human Review Template

The LLM judge is supplemental evidence, not absolute ground truth.

A blinded review file is exported with `Model A` and `Model B` labels so reviewers can manually score:

- Correctness (1–5)
- Faithfulness (1–5)
- Arabic Quality (1–5)
- Hallucination (0/1)
- Notes

In [29]:
human_review = eval_df[
    [
        "query_id",
        "query_type",
        "query",
        "gold_law",
        "gold_article",
        "gold_text",
        "augmented_context",
        "model",
        "answer_ar",
    ]
].copy()

blind_map = {
    "ALLaM-7B-Instruct-preview": "Model A",
    "Qwen3-8B": "Model B",
}

human_review["blind_model"] = human_review["model"].map(blind_map)
human_review = human_review.drop(columns=["model"])

human_review["Human_Correctness_1_5"] = ""
human_review["Human_Faithfulness_1_5"] = ""
human_review["Human_Arabic_Quality_1_5"] = ""
human_review["Human_Hallucination_0_1"] = ""
human_review["Human_Notes"] = ""

human_review = human_review.sample(
    frac=1,
    random_state=SEED,
).reset_index(drop=True)

HUMAN_REVIEW_PATH = WORK_DIR / "SANAD_Stage3_Human_Review_Blinded.csv"

human_review.to_csv(
    HUMAN_REVIEW_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", HUMAN_REVIEW_PATH)
display(human_review.head())

Saved: /content/drive/MyDrive/SANAD_Stage3/SANAD_Stage3_Human_Review_Blinded.csv


,query_id,query_type,query,gold_law,gold_article,gold_text,augmented_context,answer_ar,blind_model,Human_Correctness_1_5,Human_Faithfulness_1_5,Human_Arabic_Quality_1_5,Human_Hallucination_0_1,Human_Notes
0,Q045,informal,أقدر أرافع بنفسي قدام الدائرة التجارية بالمحكم...,اللائحة التنفيذية لنظام المحاكم التجارية,المادة الرابعة والخمسون,المادة الرابعة والخمسون : الترافع منمحام\nفي ج...,User Query:\nأقدر أرافع بنفسي قدام الدائرة الت...,يجب أن تكون المرافعة وتقديم أي مذكرات لدى الدا...,Model A,,,,,
1,Q048,informal,إذا تصالحنا بعد ما سجلنا القضية، وش يصير؟,اللائحة التنفيذية لنظام المحاكم التجارية,المادة الحادية والستون,المادة الحادية والستون : المصالحة بعدالقيد\nإذ...,User Query:\nإذا تصالحنا بعد ما سجلنا القضية، ...,إذا تصالح الأطراف بعد قيد القضية، يُثبت ما اتف...,Model A,,,,,
2,Q005,formal_paraphrase,كيف يتم تحديد قيمة التكاليف القضائية بالنسبة إ...,نظام التكاليف القضائية,المادة الثالثة,المادة الثالثة\nتفرض تكاليف قضائية على الدعوى ...,User Query:\nكيف يتم تحديد قيمة التكاليف القضا...,تُقدّر التكاليف القضائية للدعوى المحددة القيمة...,Model A,,,,,
3,Q056,formal_paraphrase,ما المتطلبات المتعلقة بالارتباط بين الطلبات ال...,اللائحة التنفيذية لنظام المحاكم التجارية,المادة السابعة والسبعون,المادة السابعة والسبعون : بيانات صحيفةالدعوى\n...,User Query:\nما المتطلبات المتعلقة بالارتباط ب...,يجب أن تتضمن صحيفة الدعوى بيان الارتباط بين ال...,Model A,,,,,
4,Q027,informal,وش لازم أكتب في الإخطار اللي أرسله للطرف الثاني؟,اللائحة التنفيذية لنظام المحاكم التجارية,المادة السبعون,المادة السبعون:\nيجب أن يتضمن الإخطار بيانات ا...,User Query:\nوش لازم أكتب في الإخطار اللي أرسل...,"{\n ""answer_ar"": ""يجب أن يتضمن الإخطار بيانات...",Model A,,,,,


# 27. Interpretation Rules

Use these rules in the report:

### If one model is higher in correctness, faithfulness, and citation accuracy
That provides the strongest evidence for selecting it as the final generator.

### If quality is nearly tied
Use Arabic quality and generation time as secondary factors.

### If one model is faster but less faithful
Do not select it only because of speed.

### If end-to-end citation accuracy is low but conditional citation accuracy is high
The main bottleneck is retrieval/reranking rather than generation.

### If the gold article is absent from Top-3
Analyze that case as an upstream retrieval limitation before blaming the generator.

### If a bootstrap interval crosses zero
Describe the observed difference as small or uncertain on this 60-query test set.

# 28. Final SANAD Pipeline

**User Arabic Query**  
↓  
**BGE-M3 Query Embedding**  
↓  
**Dense FAISS Retrieval — Top-10**  
↓  
**Qwen3-Reranker-0.6B**  
↓  
**Top-3 Saudi Legal Articles**  
↓  
**Selected Stage 3 Generator — ALLaM-7B-Instruct-preview OR Qwen3-8B**  
↓  
**Grounded Arabic Answer**  
↓  
**Citation Verification**  
↓  
**Final answer with Saudi law name + article number**

The generator synthesizes retrieved evidence; it does not replace retrieval.

## 29. Save all Stage 3 outputs

In [30]:
RESULTS_DIR = WORK_DIR / "final_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

eval_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Per_Query.csv",
    index=False,
    encoding="utf-8-sig",
)

automatic_summary_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Automatic_Metrics.csv",
    index=False,
    encoding="utf-8-sig",
)

final_comparison_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Model_Comparison.csv",
    index=False,
    encoding="utf-8-sig",
)

by_query_type_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_By_Query_Type.csv",
    index=False,
    encoding="utf-8-sig",
)

bootstrap_ci_df.to_csv(
    RESULTS_DIR / "SANAD_Stage3_Bootstrap_CI.csv",
    index=False,
    encoding="utf-8-sig",
)

shutil.copy(
    HUMAN_REVIEW_PATH,
    RESULTS_DIR / "SANAD_Stage3_Human_Review_Blinded.csv",
)

print("✅ All Stage 3 outputs saved in:")
print(RESULTS_DIR)

for p in sorted(RESULTS_DIR.iterdir()):
    print("-", p.name)

✅ All Stage 3 outputs saved in:
/content/drive/MyDrive/SANAD_Stage3/final_results
- SANAD_Stage3_Automatic_Metrics.csv
- SANAD_Stage3_Bootstrap_CI.csv
- SANAD_Stage3_By_Query_Type.csv
- SANAD_Stage3_Human_Review_Blinded.csv
- SANAD_Stage3_Model_Comparison.csv
- SANAD_Stage3_Per_Query.csv


# 30. Report-ready summary

This cell creates a compact summary from the actual run. It does not hard-code a winner before results exist.

In [31]:
summary_lines = [
    "SANAD — Stage 3 Generation & Citation Verification",
    "",
    "Final upstream pipeline:",
    "BGE-M3 Dense Retrieval → Dense Top-10 → Qwen3-Reranker-0.6B → Top-3",
    "",
    f"Evaluation queries: {len(context_df)}",
    f"Gold article available in final Top-3: {int(context_df['context_has_gold'].sum())}/{len(context_df)} "
    f"({context_df['context_has_gold'].mean():.2%})",
    "",
    "Generator comparison:",
]

for _, r in final_comparison_df.iterrows():
    summary_lines.extend([
        f"- {r['Model']}",
        f"  Correctness: {r['Correctness_1_5']:.3f}" if pd.notna(r["Correctness_1_5"]) else "  Correctness: N/A",
        f"  Faithfulness: {r['Faithfulness_1_5']:.3f}" if pd.notna(r["Faithfulness_1_5"]) else "  Faithfulness: N/A",
        f"  Citation Accuracy (all): {r['Citation_Accuracy_All']:.2%}",
        f"  Citation Accuracy (gold available): {r['Citation_Accuracy_ContextHasGold']:.2%}",
        f"  Citation Precision: {r['Citation_Precision_Context']:.2%}",
        f"  Hallucination Rate: {r['Hallucination_Rate']:.2%}" if pd.notna(r["Hallucination_Rate"]) else "  Hallucination Rate: N/A",
        f"  Arabic Quality: {r['Arabic_Quality_1_5']:.3f}" if pd.notna(r["Arabic_Quality_1_5"]) else "  Arabic Quality: N/A",
        f"  Avg Generation Time: {r['Avg_Generation_Time_s']:.3f}s",
        "",
    ])

SUMMARY_PATH = RESULTS_DIR / "SANAD_Stage3_Report_Summary.txt"
SUMMARY_PATH.write_text("\n".join(summary_lines), encoding="utf-8")

print("\n".join(summary_lines))
print("\nSaved:", SUMMARY_PATH)

SANAD — Stage 3 Generation & Citation Verification

Final upstream pipeline:
BGE-M3 Dense Retrieval → Dense Top-10 → Qwen3-Reranker-0.6B → Top-3

Evaluation queries: 60
Gold article available in final Top-3: 54/60 (90.00%)

Generator comparison:
- ALLaM-7B-Instruct-preview
  Correctness: 4.100
  Faithfulness: 3.717
  Citation Accuracy (all): 65.00%
  Citation Accuracy (gold available): 72.22%
  Citation Precision: 81.94%
  Hallucination Rate: 5.00%
  Arabic Quality: 4.267
  Avg Generation Time: 9.572s

- Qwen3-8B
  Correctness: 3.933
  Faithfulness: 3.467
  Citation Accuracy (all): 68.33%
  Citation Accuracy (gold available): 75.93%
  Citation Precision: 85.00%
  Hallucination Rate: 8.33%
  Arabic Quality: 4.133
  Avg Generation Time: 15.592s


Saved: /content/drive/MyDrive/SANAD_Stage3/final_results/SANAD_Stage3_Report_Summary.txt


## 31. Download final results ZIP

In [32]:
from google.colab import files

ZIP_BASE = "/content/SANAD_Stage3_Results"

shutil.make_archive(
    ZIP_BASE,
    "zip",
    root_dir=str(RESULTS_DIR),
)

files.download(ZIP_BASE + ".zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>